# NexaTel Customer Churn Analytics
## Phase 1 — Data Cleaning & Quality Check

**Analyst:** Kashif Siddique  
**Deliverable:** `Phase1_DataCleaning_KashifSiddique.ipynb`

### Phase 1 objectives
This notebook follows the NexaTel PRD and completes the required Phase 1 work:

- Load all 24 CSV files.
- Profile row count, column count, and data types for every table.
- Report missing values and missing-value percentages.
- Check duplicate rows and duplicate primary keys.
- Validate date and numeric data types.
- Flag `data_gb_used` outliers and negative invoice totals.
- Verify referential integrity.
- Run all PRD-specific checks for billing, customers, payments, complaints, and subscriptions.
- Compare observed findings with `data_quality_issue_log.csv`.
- Clean recoverable issues without overwriting the raw source files.
- Re-run validation after cleaning.
- Produce a written one-page Data Quality Report.

> **Important:** Raw data is treated as read-only. Cleaning is performed on copies only.


## 1. Setup and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 160)

# Portable folder setup:
# Local project structure: notebooks/ + data/raw/ + data/cleaned/ + outputs/tables/
LOCAL_RAW = Path("../data/raw")
if LOCAL_RAW.exists():
    RAW_DIR = LOCAL_RAW
    CLEAN_DIR = Path("../data/cleaned")
    OUTPUT_DIR = Path("../outputs/tables")
else:
    # ChatGPT workspace fallback
    RAW_DIR = Path("/mnt/data")
    CLEAN_DIR = Path("/mnt/data/nexatel_cleaned")
    OUTPUT_DIR = Path("/mnt/data/nexatel_outputs")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset snapshot: operational data runs through March 2026.
# The reference date is also consistent with the DOB/age values in customers.csv.
REFERENCE_DATE = pd.Timestamp("2026-04-01")
BILLING_CUTOFF = pd.Timestamp("2026-03-31")

print("RAW_DIR   :", RAW_DIR.resolve())
print("CLEAN_DIR :", CLEAN_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


RAW_DIR   : /mnt/data
CLEAN_DIR : /mnt/data/nexatel_cleaned
OUTPUT_DIR: /mnt/data/nexatel_outputs


## 2. Load all 24 CSV files

In [2]:
expected_tables = {
    "billing", "cities", "complaints", "contracts", "customer_feedback",
    "customers", "data_quality_issue_log", "devices", "employees",
    "marketing_campaigns", "network_quality", "payments", "plan_history",
    "plans", "recharges", "regions", "retention_campaigns", "states",
    "stores", "subscriptions", "support_tickets", "usage_data",
    "usage_sms", "usage_voice"
}

csv_files = sorted(
    p for p in RAW_DIR.glob("*.csv")
    if p.stem in expected_tables
)

dfs = {file.stem: pd.read_csv(file, low_memory=False) for file in csv_files}

assert set(dfs) == expected_tables, (
    f"Expected 24 NexaTel tables but found {len(dfs)}. "
    f"Missing: {sorted(expected_tables - set(dfs))}"
)

total_rows = sum(len(df) for df in dfs.values())

print(f"Tables loaded: {len(dfs)}")
print(f"Total rows: {total_rows:,}")
print("\nTables:")
print(", ".join(sorted(dfs)))


Tables loaded: 24
Total rows: 756,952

Tables:
billing, cities, complaints, contracts, customer_feedback, customers, data_quality_issue_log, devices, employees, marketing_campaigns, network_quality, payments, plan_history, plans, recharges, regions, retention_campaigns, states, stores, subscriptions, support_tickets, usage_data, usage_sms, usage_voice


**Baseline validation:** the supplied NexaTel dataset contains 24 tables and 756,952 rows in total.

## 3. Basic profile — row count, column count, and data types

In [3]:
primary_keys = {
    "customers": "customer_id",
    "subscriptions": "subscription_id",
    "plans": "plan_id",
    "plan_history": "history_id",
    "contracts": "contract_id",
    "devices": "device_id",
    "billing": "invoice_id",
    "payments": "payment_id",
    "recharges": "recharge_id",
    "usage_voice": "usage_id",
    "usage_sms": "usage_id",
    "usage_data": "usage_id",
    "network_quality": "record_id",
    "support_tickets": "ticket_id",
    "complaints": "complaint_id",
    "customer_feedback": "feedback_id",
    "retention_campaigns": "campaign_id",
    "marketing_campaigns": "campaign_id",
    "employees": "employee_id",
    "stores": "store_id",
    "regions": "region_id",
    "cities": "city_id",
    "states": "state_id",
}

profile_rows = []
dtype_rows = []

for table, df in dfs.items():
    pk = primary_keys.get(table)

    profile_rows.append({
        "table": table,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_primary_key_rows": (
            int(df.duplicated(subset=[pk]).sum()) if pk else np.nan
        ),
    })

    for column in df.columns:
        dtype_rows.append({
            "table": table,
            "column": column,
            "pandas_dtype": str(df[column].dtype),
            "non_null_rows": int(df[column].notna().sum()),
            "null_rows": int(df[column].isna().sum()),
        })

dataset_profile = (
    pd.DataFrame(profile_rows)
    .sort_values("table")
    .reset_index(drop=True)
)

column_dtypes = (
    pd.DataFrame(dtype_rows)
    .sort_values(["table", "column"])
    .reset_index(drop=True)
)

display(dataset_profile)
display(column_dtypes)


,table,rows,columns,missing_cells,exact_duplicate_rows,duplicate_primary_key_rows
0,billing,115313,9,0,344,344.0
1,cities,68,8,0,0,0.0
2,complaints,16584,7,7348,0,0.0
3,contracts,10607,9,0,0,0.0
4,customer_feedback,28499,7,0,0,0.0
5,customers,19076,25,17639,76,76.0
6,data_quality_issue_log,20,3,0,0,NaN
7,devices,19000,8,0,0,0.0
8,employees,800,9,0,0,0.0
9,marketing_campaigns,40,9,0,0,0.0


,table,column,pandas_dtype,non_null_rows,null_rows
0,billing,base_amount_inr,float64,115313,0
1,billing,billing_date,object,115313,0
2,billing,billing_period_month,object,115313,0
3,billing,customer_id,object,115313,0
4,billing,due_date,object,115313,0
5,billing,gst_amount_inr,float64,115313,0
6,billing,invoice_id,object,115313,0
7,billing,payment_status,object,115313,0
8,billing,total_amount_inr,float64,115313,0
9,cities,city_id,object,68,0


In [4]:
dataset_profile.to_csv(OUTPUT_DIR / "phase1_dataset_profile.csv", index=False)
column_dtypes.to_csv(OUTPUT_DIR / "phase1_column_dtypes.csv", index=False)


## 4. Missing-value analysis

In [5]:
missing_rows = []

for table, df in dfs.items():
    for col in df.columns:
        count = int(df[col].isna().sum())
        if count > 0:
            missing_rows.append({
                "table": table,
                "column": col,
                "missing_count": count,
                "missing_pct": round(count / len(df) * 100, 2),
            })

missing_summary = (
    pd.DataFrame(missing_rows)
    .sort_values(["table", "missing_pct"], ascending=[True, False])
    .reset_index(drop=True)
)

display(missing_summary)
missing_summary.to_csv(OUTPUT_DIR / "phase1_missing_values.csv", index=False)


,table,column,missing_count,missing_pct
0,complaints,resolution_date,7348,44.31
1,customers,churn_date,15560,81.57
2,customers,annual_income_inr,954,5.00
3,customers,email,745,3.91
4,customers,address,380,1.99
5,subscriptions,end_date,17017,82.92
6,subscriptions,plan_id,41,0.20
7,support_tickets,resolution_hours,13366,42.45
8,support_tickets,resolution_date,13366,42.45


### Missing-value interpretation

Null values are **not automatically errors**. Examples of expected nulls include:

- `customers.churn_date` for active customers.
- `subscriptions.end_date` for active subscriptions.
- `support_tickets.resolution_date` and `resolution_hours` for unresolved/open tickets.
- `complaints.resolution_date` for unresolved complaints.
- `payments.invoice_id` is documented as nullable.
- `subscriptions.plan_id` is documented as nullable, although the PRD asks us to count these rows.


## 5. Duplicate analysis

In [6]:
duplicate_summary = dataset_profile.loc[
    (dataset_profile["exact_duplicate_rows"] > 0)
    | (dataset_profile["duplicate_primary_key_rows"].fillna(0) > 0),
    ["table", "exact_duplicate_rows", "duplicate_primary_key_rows"]
].copy()

display(duplicate_summary)


,table,exact_duplicate_rows,duplicate_primary_key_rows
0,billing,344,344.0
5,customers,76,76.0
11,payments,276,276.0


The duplicate check is performed both at the **full-row level** and at the **documented primary-key level**. This directly covers the PRD checks for duplicate `customer_id` values in `customers.csv` and duplicate `invoice_id` values in `billing.csv`.

## 6. Data-type validation

### 6.1 Date validation

In [7]:
# DD-MM-YYYY date columns documented by the schema/data dictionary.
date_columns = {
    "customers": ["date_of_birth", "acquisition_date", "churn_date"],
    "subscriptions": ["start_date", "end_date"],
    "plan_history": ["change_date"],
    "contracts": ["start_date", "end_date"],
    "devices": ["purchase_date"],
    "billing": ["billing_date", "due_date"],
    "payments": ["payment_date"],
    "recharges": ["recharge_date"],
    "support_tickets": ["created_date", "resolution_date"],
    "complaints": ["complaint_date", "resolution_date"],
    "customer_feedback": ["feedback_date"],
    "retention_campaigns": ["contact_date"],
    "marketing_campaigns": ["start_date", "end_date"],
    "employees": ["hire_date"],
    "stores": ["opened_date"],
}

month_columns = {
    "billing": ["billing_period_month"],
    "usage_voice": ["usage_month"],
    "usage_sms": ["usage_month"],
    "usage_data": ["usage_month"],
    "network_quality": ["month"],
}

date_validation_rows = []

for table, cols in date_columns.items():
    for col in cols:
        s = dfs[table][col]
        parsed = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
        invalid = s.notna() & parsed.isna()

        date_validation_rows.append({
            "table": table,
            "column": col,
            "expected_format": "DD-MM-YYYY",
            "non_null_rows": int(s.notna().sum()),
            "invalid_format_rows": int(invalid.sum()),
        })

for table, cols in month_columns.items():
    for col in cols:
        s = dfs[table][col]
        parsed = pd.to_datetime(s, format="%m-%Y", errors="coerce")
        invalid = s.notna() & parsed.isna()

        date_validation_rows.append({
            "table": table,
            "column": col,
            "expected_format": "MM-YYYY",
            "non_null_rows": int(s.notna().sum()),
            "invalid_format_rows": int(invalid.sum()),
        })

date_validation = pd.DataFrame(date_validation_rows)
display(date_validation)
date_validation.to_csv(OUTPUT_DIR / "phase1_date_validation.csv", index=False)


,table,column,expected_format,non_null_rows,invalid_format_rows
0,customers,date_of_birth,DD-MM-YYYY,19076,0
1,customers,acquisition_date,DD-MM-YYYY,19076,383
2,customers,churn_date,DD-MM-YYYY,3516,0
3,subscriptions,start_date,DD-MM-YYYY,20523,0
4,subscriptions,end_date,DD-MM-YYYY,3506,0
5,plan_history,change_date,DD-MM-YYYY,19349,0
6,contracts,start_date,DD-MM-YYYY,10607,0
7,contracts,end_date,DD-MM-YYYY,10607,0
8,devices,purchase_date,DD-MM-YYYY,19000,0
9,billing,billing_date,DD-MM-YYYY,115313,0


`customers.acquisition_date` is expected to show strict-format failures because the Data Dictionary states that some records use other date formats. These rows are handled explicitly during cleaning rather than silently accepted during QC.

### 6.2 Numeric/amount validation

In [8]:
numeric_columns = {
    "customers": ["age", "annual_income_inr", "tenure_months"],
    "subscriptions": ["monthly_charge_inr"],
    "plans": ["monthly_charge", "data_quota_gb", "voice_minutes", "sms_count"],
    "contracts": ["contract_length_months"],
    "devices": ["device_price_inr"],
    "billing": ["base_amount_inr", "gst_amount_inr", "total_amount_inr"],
    "payments": ["amount_inr"],
    "recharges": ["amount_inr"],
    "usage_voice": ["voice_minutes_used", "outgoing_calls", "incoming_calls"],
    "usage_sms": ["sms_sent", "sms_received"],
    "usage_data": ["data_gb_used", "data_sessions"],
    "network_quality": [
        "avg_signal_strength_dbm", "dropped_call_rate_pct",
        "avg_download_speed_mbps", "network_downtime_hours"
    ],
    "support_tickets": ["resolution_hours"],
    "customer_feedback": ["csat_score", "nps_score"],
    "marketing_campaigns": ["budget_inr", "customers_reached", "conversions"],
    "data_quality_issue_log": ["rows_affected"],
}

numeric_validation_rows = []

for table, cols in numeric_columns.items():
    for col in cols:
        s = dfs[table][col]
        converted = pd.to_numeric(s, errors="coerce")
        invalid = s.notna() & converted.isna()

        numeric_validation_rows.append({
            "table": table,
            "column": col,
            "raw_dtype": str(s.dtype),
            "non_null_rows": int(s.notna().sum()),
            "non_numeric_rows": int(invalid.sum()),
        })

numeric_validation = pd.DataFrame(numeric_validation_rows)
display(numeric_validation)
numeric_validation.to_csv(OUTPUT_DIR / "phase1_numeric_validation.csv", index=False)


,table,column,raw_dtype,non_null_rows,non_numeric_rows
0,customers,age,int64,19076,0
1,customers,annual_income_inr,float64,18122,0
2,customers,tenure_months,int64,19076,0
3,subscriptions,monthly_charge_inr,float64,20523,0
4,plans,monthly_charge,int64,25,0
5,plans,data_quota_gb,int64,25,0
6,plans,voice_minutes,int64,25,0
7,plans,sms_count,int64,25,0
8,contracts,contract_length_months,int64,10607,0
9,devices,device_price_inr,int64,19000,0


## 7. PRD-specific quality checks

In [9]:
# De-duplicate copies for issue counting so repeated identical rows do not
# inflate the same underlying quality defect.
customers_u = dfs["customers"].drop_duplicates().copy()
billing_u = dfs["billing"].drop_duplicates().copy()
payments_u = dfs["payments"].drop_duplicates().copy()

cities = dfs["cities"].copy()


### 7.1 Customers — ages, PIN codes, phone numbers, email, city, dates

In [10]:
# --- Age checks ---
impossible_age_range = (customers_u["age"] < 0) | (customers_u["age"] > 120)

dob = pd.to_datetime(
    customers_u["date_of_birth"], format="%d-%m-%Y", errors="coerce"
)
expected_age = (
    REFERENCE_DATE.year
    - dob.dt.year
    - (
        (REFERENCE_DATE.month < dob.dt.month)
        | (
            (REFERENCE_DATE.month == dob.dt.month)
            & (REFERENCE_DATE.day < dob.dt.day)
        )
    ).astype(int)
)
age_dob_mismatch = customers_u["age"].ne(expected_age)

# --- PIN code ---
pin_s = (
    customers_u["pin_code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)
invalid_pin = ~pin_s.str.match(r"^\d{6}$", na=False)

# --- Phone number ---
phone_s = (
    customers_u["phone_number"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)
invalid_phone = ~phone_s.str.match(r"^91[6-9]\d{9}$", na=False)

# --- Email ---
email_s = customers_u["email"].astype("string")
invalid_email = (
    customers_u["email"].notna()
    & ~email_s.str.match(r"^[^\s@]+@[^\s@]+\.[^\s@]+$", na=False)
)

# --- City consistency ---
city_lookup = cities.set_index("city_id")["city_name"]
canonical_city = customers_u["city_id"].map(city_lookup)

misspelled_city = (
    customers_u["city_name"].str.lower()
    != canonical_city.str.lower()
)
capitalisation_issue = (
    customers_u["city_name"].str.lower().eq(canonical_city.str.lower())
    & customers_u["city_name"].ne(canonical_city)
)

# --- Mixed acquisition-date format ---
acq_strict = pd.to_datetime(
    customers_u["acquisition_date"],
    format="%d-%m-%Y",
    errors="coerce",
)
mixed_acquisition_date = (
    customers_u["acquisition_date"].notna()
    & acq_strict.isna()
)

customer_quality = pd.DataFrame([
    ["Impossible age (<0 or >120)", int(impossible_age_range.sum())],
    ["Age inconsistent with DOB/reference date", int(age_dob_mismatch.sum())],
    ["Invalid PIN code", int(invalid_pin.sum())],
    ["Invalid phone number", int(invalid_phone.sum())],
    ["Invalid non-null email", int(invalid_email.sum())],
    ["Missing email", int(customers_u["email"].isna().sum())],
    ["Missing address", int(customers_u["address"].isna().sum())],
    ["Missing annual income", int(customers_u["annual_income_inr"].isna().sum())],
    ["Misspelled city", int(misspelled_city.sum())],
    ["Capitalisation inconsistency", int(capitalisation_issue.sum())],
    ["Mixed-format acquisition date", int(mixed_acquisition_date.sum())],
], columns=["check", "rows_affected"])

display(customer_quality)


,check,rows_affected
0,Impossible age (<0 or >120),23
1,Age inconsistent with DOB/reference date,38
2,Invalid PIN code,190
3,Invalid phone number,228
4,Invalid non-null email,285
5,Missing email,744
6,Missing address,380
7,Missing annual income,950
8,Misspelled city,380
9,Capitalisation inconsistency,560


### 7.2 Billing — future dates, negative totals, GST

In [11]:
billing_date = pd.to_datetime(
    billing_u["billing_date"], format="%d-%m-%Y", errors="coerce"
)

future_billing_date = billing_date > BILLING_CUTOFF
negative_invoice = billing_u["total_amount_inr"] < 0

expected_gst = (billing_u["base_amount_inr"] * 0.18).round(2)
incorrect_gst = (billing_u["gst_amount_inr"] - expected_gst).abs() > 0.01

billing_quality = pd.DataFrame([
    ["Future billing_date", int(future_billing_date.sum())],
    ["Negative total_amount_inr", int(negative_invoice.sum())],
    ["Incorrect GST amount", int(incorrect_gst.sum())],
    ["Duplicate invoices", int(dfs["billing"].duplicated("invoice_id").sum())],
], columns=["check", "rows_affected"])

display(billing_quality)


,check,rows_affected
0,Future billing_date,229
1,Negative total_amount_inr,229
2,Incorrect GST amount,1149
3,Duplicate invoices,344


### 7.3 Payments — broken invoice references

In [12]:
valid_invoice_ids = set(dfs["billing"]["invoice_id"])

broken_payment_invoice_fk = (
    payments_u["invoice_id"].notna()
    & ~payments_u["invoice_id"].isin(valid_invoice_ids)
)

payments_quality = pd.DataFrame([
    ["Broken non-null invoice_id foreign key", int(broken_payment_invoice_fk.sum())],
    ["Duplicate payments", int(dfs["payments"].duplicated("payment_id").sum())],
], columns=["check", "rows_affected"])

display(payments_quality)


,check,rows_affected
0,Broken non-null invoice_id foreign key,184
1,Duplicate payments,276


### 7.4 Complaints — unresolved share

In [13]:
complaints = dfs["complaints"]

unresolved_complaints = int(
    (complaints["status"].str.strip().str.lower() == "unresolved").sum()
)
unresolved_share = (
    unresolved_complaints / len(complaints) * 100
)

print(f"Unresolved complaints: {unresolved_complaints:,} / {len(complaints):,}")
print(f"Unresolved complaint share: {unresolved_share:.2f}%")


Unresolved complaints: 7,348 / 16,584
Unresolved complaint share: 44.31%


### 7.5 Subscriptions — null plan ID and invalid status

In [14]:
subscriptions = dfs["subscriptions"]

null_plan_id = subscriptions["plan_id"].isna()
valid_subscription_statuses = {"Active", "Terminated"}
invalid_subscription_status = ~subscriptions["subscription_status"].isin(
    valid_subscription_statuses
)

subscription_quality = pd.DataFrame([
    ["Null plan_id", int(null_plan_id.sum())],
    ["Invalid subscription_status", int(invalid_subscription_status.sum())],
], columns=["check", "rows_affected"])

display(subscription_quality)

print("\nRaw subscription status values:")
display(subscriptions["subscription_status"].value_counts(dropna=False).rename("rows"))


,check,rows_affected
0,Null plan_id,41
1,Invalid subscription_status,102



Raw subscription status values:


subscription_status
Active           16930
Terminated        3491
UNKNOWN_STATE      102
Name: rows, dtype: int64

### 7.6 `data_gb_used` outliers

In [15]:
usage = dfs["usage_data"].copy()
data_gb = usage["data_gb_used"]

# Statistical outlier flag required by the PRD (IQR rule).
q1 = data_gb.quantile(0.25)
q3 = data_gb.quantile(0.75)
iqr = q3 - q1
iqr_lower = q1 - 1.5 * iqr
iqr_upper = q3 + 1.5 * iqr
iqr_outlier = (data_gb < iqr_lower) | (data_gb > iqr_upper)

# The schema documents data_gb_used as DECIMAL(...,2).
# A smaller set of values contains precision beyond 2 decimals and corresponds
# to the deliberately injected corrupted outlier records in the DQ issue log.
precision_anomaly = (
    data_gb.sub(data_gb.round(2)).abs() > 1e-9
)

outlier_summary = pd.DataFrame([
    ["IQR lower bound", iqr_lower],
    ["IQR upper bound", iqr_upper],
    ["Statistical IQR outlier rows", int(iqr_outlier.sum())],
    ["Confirmed corrupted precision-anomaly rows", int(precision_anomaly.sum())],
    ["Maximum raw data_gb_used", data_gb.max()],
], columns=["metric", "value"])

display(outlier_summary)


,metric,value
0,IQR lower bound,-20.470000
1,IQR upper bound,64.490000
2,Statistical IQR outlier rows,4480.000000
3,Confirmed corrupted precision-anomaly rows,171.000000
4,Maximum raw data_gb_used,22172.700708


**Cleaning decision:** IQR flags are retained as analytical flags because unusually high usage can be legitimate. Only the precision-anomaly records that match the deliberately injected data-quality issue are treated as corrupted values during cleaning. This avoids deleting genuine heavy users merely because they are statistically unusual.

## 8. Referential-integrity validation

In [16]:
# Required PRD customer_id checks plus additional useful foreign-key checks.
customer_ids = set(customers_u["customer_id"])
plan_ids = set(dfs["plans"]["plan_id"])
invoice_ids = set(dfs["billing"]["invoice_id"])
employee_ids = set(dfs["employees"]["employee_id"])
city_ids = set(dfs["cities"]["city_id"])

fk_checks = [
    # PRD-required customer relationships
    ("subscriptions.customer_id -> customers.customer_id",
     dfs["subscriptions"]["customer_id"], customer_ids),
    ("billing.customer_id -> customers.customer_id",
     dfs["billing"]["customer_id"], customer_ids),
    ("usage_voice.customer_id -> customers.customer_id",
     dfs["usage_voice"]["customer_id"], customer_ids),
    ("usage_sms.customer_id -> customers.customer_id",
     dfs["usage_sms"]["customer_id"], customer_ids),
    ("usage_data.customer_id -> customers.customer_id",
     dfs["usage_data"]["customer_id"], customer_ids),

    # Additional integrity checks
    ("network_quality.customer_id -> customers.customer_id",
     dfs["network_quality"]["customer_id"], customer_ids),
    ("payments.customer_id -> customers.customer_id",
     dfs["payments"]["customer_id"], customer_ids),
    ("complaints.customer_id -> customers.customer_id",
     dfs["complaints"]["customer_id"], customer_ids),
    ("support_tickets.customer_id -> customers.customer_id",
     dfs["support_tickets"]["customer_id"], customer_ids),
    ("payments.invoice_id -> billing.invoice_id",
     dfs["payments"]["invoice_id"].dropna(), invoice_ids),
    ("subscriptions.plan_id -> plans.plan_id",
     dfs["subscriptions"]["plan_id"].dropna(), plan_ids),
    ("support_tickets.assigned_employee_id -> employees.employee_id",
     dfs["support_tickets"]["assigned_employee_id"], employee_ids),
    ("network_quality.city_id -> cities.city_id",
     dfs["network_quality"]["city_id"], city_ids),
]

ri_rows = []

for relationship, values, valid_values in fk_checks:
    broken = ~values.isin(valid_values)

    ri_rows.append({
        "relationship": relationship,
        "checked_rows": len(values),
        "broken_rows": int(broken.sum()),
        "broken_unique_values": int(values[broken].nunique()),
    })

referential_integrity = pd.DataFrame(ri_rows)
display(referential_integrity)

referential_integrity.to_csv(
    OUTPUT_DIR / "phase1_referential_integrity.csv", index=False
)


,relationship,checked_rows,broken_rows,broken_unique_values
0,subscriptions.customer_id -> customers.custome...,20523,0,0
1,billing.customer_id -> customers.customer_id,115313,0,0
2,usage_voice.customer_id -> customers.customer_id,57000,0,0
3,usage_sms.customer_id -> customers.customer_id,57000,0,0
4,usage_data.customer_id -> customers.customer_id,57000,0,0
5,network_quality.customer_id -> customers.custo...,57000,0,0
6,payments.customer_id -> customers.customer_id,92362,0,0
7,complaints.customer_id -> customers.customer_id,16584,0,0
8,support_tickets.customer_id -> customers.custo...,31490,0,0
9,payments.invoice_id -> billing.invoice_id,92362,184,184


## 9. Compare observed issues with `data_quality_issue_log.csv`

In [17]:
dq_log = dfs["data_quality_issue_log"].copy()

# The raw CSV names this field 'table', whereas the Data Dictionary/schema
# document it as 'issue_table'. Standardize it for the cleaned version.
if "table" in dq_log.columns and "issue_table" not in dq_log.columns:
    dq_log = dq_log.rename(columns={"table": "issue_table"})

observed_map = {
    ("customers", "missing_email"): int(customers_u["email"].isna().sum()),
    ("customers", "missing_address"): int(customers_u["address"].isna().sum()),
    ("customers", "missing_income"): int(customers_u["annual_income_inr"].isna().sum()),
    ("customers", "invalid_phone"): int(invalid_phone.sum()),
    ("customers", "invalid_email"): int(invalid_email.sum()),
    ("customers", "incorrect_pin"): int(invalid_pin.sum()),
    ("customers", "misspelled_city"): int(misspelled_city.sum()),
    ("customers", "inconsistent_capitalisation"): int(capitalisation_issue.sum()),
    ("customers", "impossible_age"): int(age_dob_mismatch.sum()),
    ("customers", "mixed_date_format"): int(mixed_acquisition_date.sum()),
    ("customers", "duplicate_customers"): int(dfs["customers"].duplicated("customer_id").sum()),

    ("billing", "incorrect_gst"): int(incorrect_gst.sum()),
    ("billing", "negative_invoice_amount"): int(negative_invoice.sum()),
    ("billing", "future_dates"): int(future_billing_date.sum()),
    ("billing", "duplicate_invoices"): int(dfs["billing"].duplicated("invoice_id").sum()),

    ("payments", "broken_foreign_key"): int(broken_payment_invoice_fk.sum()),
    ("payments", "duplicate_payments"): int(dfs["payments"].duplicated("payment_id").sum()),

    ("subscriptions", "null_foreign_key"): int(null_plan_id.sum()),
    ("subscriptions", "invalid_category"): int(invalid_subscription_status.sum()),

    ("usage_data", "outlier_values"): int(precision_anomaly.sum()),
}

dq_comparison = dq_log.copy()
dq_comparison["observed_rows"] = [
    observed_map.get((r.issue_table, r.issue_type), np.nan)
    for r in dq_comparison.itertuples()
]
dq_comparison["difference"] = (
    dq_comparison["observed_rows"] - dq_comparison["rows_affected"]
)
dq_comparison["match"] = dq_comparison["difference"].eq(0)

display(dq_comparison)
dq_comparison.to_csv(
    OUTPUT_DIR / "phase1_dq_log_comparison.csv", index=False
)


,issue_table,issue_type,rows_affected,observed_rows,difference,match
0,customers,missing_email,760,744,-16,False
1,customers,missing_address,380,380,0,True
2,customers,missing_income,950,950,0,True
3,customers,invalid_phone,228,228,0,True
4,customers,invalid_email,285,285,0,True
5,customers,incorrect_pin,190,190,0,True
6,customers,misspelled_city,380,380,0,True
7,customers,inconsistent_capitalisation,570,560,-10,False
8,customers,impossible_age,38,38,0,True
9,customers,mixed_date_format,380,380,0,True


### DQ-log interpretation

The issue log is a reference list of deliberately injected defects. The audit reports what is **actually observable** in the current raw CSV files. A small difference can occur where injected issues overlap (for example, an invalid value may overwrite an earlier missing-value injection). We document these differences rather than forcing the observed counts to equal the log.

## 10. Cleaning policy

The cleaning rules are intentionally conservative:

1. Remove exact/primary-key duplicate records where the duplicated row is identical.
2. Standardize documented mixed date formats.
3. Correct city names using `city_id` and the canonical `cities` lookup.
4. Recalculate age from DOB when the stored age is inconsistent.
5. Convert malformed emails/phones/PINs to missing when the original value cannot be reconstructed safely.
6. Correct GST from the documented 18% rule and recompute invoice totals from billing components.
7. Repair corrupted future billing dates using the validated relationship `due_date = billing_date + 15 days`.
8. Preserve valid nullable fields rather than inventing data.
9. Replace broken non-null payment invoice links with null because `payments.invoice_id` is documented as nullable.
10. Infer invalid subscription status only where `end_date` provides a clear rule.
11. Flag statistical usage outliers, but set only confirmed corrupted usage values to missing.


## 11. Apply cleaning

In [18]:
clean = {name: df.copy() for name, df in dfs.items()}

# ------------------------------------------------------------------
# customers
# ------------------------------------------------------------------
c = clean["customers"].drop_duplicates(subset=["customer_id"], keep="first").copy()

# Dates
c["date_of_birth"] = pd.to_datetime(
    c["date_of_birth"], format="%d-%m-%Y", errors="coerce"
)

acq_ddmmyyyy = pd.to_datetime(
    c["acquisition_date"], format="%d-%m-%Y", errors="coerce"
)
acq_yyyymmdd = pd.to_datetime(
    c["acquisition_date"], format="%Y/%m/%d", errors="coerce"
)
c["acquisition_date"] = acq_ddmmyyyy.fillna(acq_yyyymmdd)

c["churn_date"] = pd.to_datetime(
    c["churn_date"], format="%d-%m-%Y", errors="coerce"
)

# Canonical city from lookup
canonical_city_map = clean["cities"].set_index("city_id")["city_name"]
c["city_name"] = c["city_id"].map(canonical_city_map).fillna(c["city_name"])

# Recalculate age from DOB
dob = c["date_of_birth"]
clean_age = (
    REFERENCE_DATE.year
    - dob.dt.year
    - (
        (REFERENCE_DATE.month < dob.dt.month)
        | (
            (REFERENCE_DATE.month == dob.dt.month)
            & (REFERENCE_DATE.day < dob.dt.day)
        )
    ).astype(int)
)
c["age"] = clean_age.astype("Int64")

# Invalid email -> missing
email_ok = (
    c["email"].isna()
    | c["email"].astype("string").str.match(
        r"^[^\s@]+@[^\s@]+\.[^\s@]+$", na=False
    )
)
c.loc[~email_ok, "email"] = pd.NA

# Invalid phone -> missing
c["phone_number"] = (
    c["phone_number"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)
phone_ok = c["phone_number"].str.match(r"^91[6-9]\d{9}$", na=False)
c.loc[~phone_ok, "phone_number"] = pd.NA

# Invalid PIN -> missing
c["pin_code"] = (
    c["pin_code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)
pin_ok = c["pin_code"].str.match(r"^\d{6}$", na=False)
c.loc[~pin_ok, "pin_code"] = pd.NA

clean["customers"] = c

# ------------------------------------------------------------------
# billing
# ------------------------------------------------------------------
b = clean["billing"].drop_duplicates(subset=["invoice_id"], keep="first").copy()

b["billing_date"] = pd.to_datetime(
    b["billing_date"], format="%d-%m-%Y", errors="coerce"
)
b["due_date"] = pd.to_datetime(
    b["due_date"], format="%d-%m-%Y", errors="coerce"
)

# Validate normal 15-day due-date relationship before using it for repair.
normal_rows = b["billing_date"] <= BILLING_CUTOFF
normal_due_delta = (
    b.loc[normal_rows, "due_date"]
    - b.loc[normal_rows, "billing_date"]
).dt.days
assert normal_due_delta.eq(15).all(), (
    "Unexpected due-date relationship detected; future-date repair stopped."
)

future_mask = b["billing_date"] > BILLING_CUTOFF
b.loc[future_mask, "billing_date"] = (
    b.loc[future_mask, "due_date"] - pd.Timedelta(days=15)
)

# Correct GST and total
b["gst_amount_inr"] = (b["base_amount_inr"] * 0.18).round(2)
b["total_amount_inr"] = (
    b["base_amount_inr"] + b["gst_amount_inr"]
).round(2)

clean["billing"] = b

# ------------------------------------------------------------------
# payments
# ------------------------------------------------------------------
p = clean["payments"].drop_duplicates(subset=["payment_id"], keep="first").copy()
p["payment_date"] = pd.to_datetime(
    p["payment_date"], format="%d-%m-%Y", errors="coerce"
)

valid_clean_invoice_ids = set(clean["billing"]["invoice_id"])
bad_invoice_link = (
    p["invoice_id"].notna()
    & ~p["invoice_id"].isin(valid_clean_invoice_ids)
)

# invoice_id is documented as nullable, so preserve the payment row and
# clear only the broken relationship.
p.loc[bad_invoice_link, "invoice_id"] = pd.NA
clean["payments"] = p

# ------------------------------------------------------------------
# subscriptions
# ------------------------------------------------------------------
s = clean["subscriptions"].copy()
s["start_date"] = pd.to_datetime(
    s["start_date"], format="%d-%m-%Y", errors="coerce"
)
s["end_date"] = pd.to_datetime(
    s["end_date"], format="%d-%m-%Y", errors="coerce"
)

invalid_status_mask = ~s["subscription_status"].isin(
    {"Active", "Terminated"}
)

s.loc[
    invalid_status_mask & s["end_date"].isna(),
    "subscription_status"
] = "Active"

s.loc[
    invalid_status_mask & s["end_date"].notna(),
    "subscription_status"
] = "Terminated"

# Null plan_id is retained: there is no guaranteed source from which to
# reconstruct the correct historical plan.
clean["subscriptions"] = s

# ------------------------------------------------------------------
# usage_data
# ------------------------------------------------------------------
u = clean["usage_data"].copy()

q1 = u["data_gb_used"].quantile(0.25)
q3 = u["data_gb_used"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

u["data_gb_used_iqr_outlier_flag"] = (
    (u["data_gb_used"] < lower)
    | (u["data_gb_used"] > upper)
)

corrupt_usage = (
    u["data_gb_used"]
    .sub(u["data_gb_used"].round(2))
    .abs()
    .gt(1e-9)
)

u["data_gb_used_corrupt_flag"] = corrupt_usage
u.loc[corrupt_usage, "data_gb_used"] = np.nan

clean["usage_data"] = u

# ------------------------------------------------------------------
# remaining date columns
# ------------------------------------------------------------------
other_date_columns = {
    "plan_history": ["change_date"],
    "contracts": ["start_date", "end_date"],
    "devices": ["purchase_date"],
    "recharges": ["recharge_date"],
    "support_tickets": ["created_date", "resolution_date"],
    "complaints": ["complaint_date", "resolution_date"],
    "customer_feedback": ["feedback_date"],
    "retention_campaigns": ["contact_date"],
    "marketing_campaigns": ["start_date", "end_date"],
    "employees": ["hire_date"],
    "stores": ["opened_date"],
}

for table, cols in other_date_columns.items():
    for col in cols:
        clean[table][col] = pd.to_datetime(
            clean[table][col],
            format="%d-%m-%Y",
            errors="coerce",
        )

# Standardize DQ-log column name to match the Data Dictionary/schema
if "table" in clean["data_quality_issue_log"].columns:
    clean["data_quality_issue_log"] = (
        clean["data_quality_issue_log"]
        .rename(columns={"table": "issue_table"})
    )


## 12. Post-cleaning validation

In [19]:
validation_rows = []

def add_validation(check, remaining):
    validation_rows.append({
        "check": check,
        "remaining_issues": int(remaining),
        "status": "PASS" if int(remaining) == 0 else "REVIEW",
    })

# Duplicates
add_validation(
    "Duplicate customers",
    clean["customers"].duplicated("customer_id").sum(),
)
add_validation(
    "Duplicate invoices",
    clean["billing"].duplicated("invoice_id").sum(),
)
add_validation(
    "Duplicate payments",
    clean["payments"].duplicated("payment_id").sum(),
)

# Customers
add_validation(
    "Invalid non-null email",
    (
        clean["customers"]["email"].notna()
        & ~clean["customers"]["email"].astype("string").str.match(
            r"^[^\s@]+@[^\s@]+\.[^\s@]+$", na=False
        )
    ).sum(),
)
add_validation(
    "Invalid non-null phone",
    (
        clean["customers"]["phone_number"].notna()
        & ~clean["customers"]["phone_number"].astype("string").str.match(
            r"^91[6-9]\d{9}$", na=False
        )
    ).sum(),
)
add_validation(
    "Invalid non-null PIN",
    (
        clean["customers"]["pin_code"].notna()
        & ~clean["customers"]["pin_code"].astype("string").str.match(
            r"^\d{6}$", na=False
        )
    ).sum(),
)
add_validation(
    "City-name mismatch",
    (
        clean["customers"]["city_name"]
        != clean["customers"]["city_id"].map(
            clean["cities"].set_index("city_id")["city_name"]
        )
    ).sum(),
)
add_validation(
    "Impossible age (<0 or >120)",
    (
        (clean["customers"]["age"] < 0)
        | (clean["customers"]["age"] > 120)
    ).sum(),
)

# Billing
add_validation(
    "Future billing dates",
    (clean["billing"]["billing_date"] > BILLING_CUTOFF).sum(),
)
add_validation(
    "Negative invoice totals",
    (clean["billing"]["total_amount_inr"] < 0).sum(),
)
add_validation(
    "Incorrect GST",
    (
        clean["billing"]["gst_amount_inr"]
        - (clean["billing"]["base_amount_inr"] * 0.18).round(2)
    ).abs().gt(0.01).sum(),
)

# Payments
add_validation(
    "Broken non-null payment invoice references",
    (
        clean["payments"]["invoice_id"].notna()
        & ~clean["payments"]["invoice_id"].isin(
            set(clean["billing"]["invoice_id"])
        )
    ).sum(),
)

# Subscriptions
add_validation(
    "Invalid subscription status",
    ~clean["subscriptions"]["subscription_status"].isin(
        {"Active", "Terminated"}
    ).sum() if False else (
        ~clean["subscriptions"]["subscription_status"].isin(
            {"Active", "Terminated"}
        )
    ).sum(),
)

# Confirmed corrupt usage values
add_validation(
    "Corrupted usage values still present",
    (
        clean["usage_data"]["data_gb_used"].notna()
        & clean["usage_data"]["data_gb_used"]
            .sub(clean["usage_data"]["data_gb_used"].round(2))
            .abs()
            .gt(1e-9)
    ).sum(),
)

post_cleaning_validation = pd.DataFrame(validation_rows)
display(post_cleaning_validation)

assert (post_cleaning_validation["status"] == "PASS").all(), (
    "One or more post-cleaning validation checks still require review."
)

print("All targeted post-cleaning checks passed.")


,check,remaining_issues,status
0,Duplicate customers,0,PASS
1,Duplicate invoices,0,PASS
2,Duplicate payments,0,PASS
3,Invalid non-null email,0,PASS
4,Invalid non-null phone,0,PASS
5,Invalid non-null PIN,0,PASS
6,City-name mismatch,0,PASS
7,Impossible age (<0 or >120),0,PASS
8,Future billing dates,0,PASS
9,Negative invoice totals,0,PASS


All targeted post-cleaning checks passed.


## 13. Export cleaned tables and QC outputs

In [20]:
for table, df in clean.items():
    df.to_csv(
        CLEAN_DIR / f"{table}_clean.csv",
        index=False,
        date_format="%Y-%m-%d",
    )

post_cleaning_validation.to_csv(
    OUTPUT_DIR / "phase1_post_cleaning_validation.csv",
    index=False,
)

print(f"Exported {len(clean)} cleaned tables to:")
print(CLEAN_DIR.resolve())


Exported 24 cleaned tables to:
/mnt/data/nexatel_cleaned


## 14. One-page Data Quality Report

In [21]:
# Key figures used in the written report
duplicate_customers = int(dfs["customers"].duplicated("customer_id").sum())
duplicate_invoices = int(dfs["billing"].duplicated("invoice_id").sum())
duplicate_payments = int(dfs["payments"].duplicated("payment_id").sum())

report_md = f"""
### NexaTel Phase 1 Data Quality Report

**Scope.** The Phase 1 audit reviewed **{len(dfs)} interconnected tables containing {total_rows:,} raw records**. The assessment covered structure, missing values, duplicates, date/numeric validity, outliers, referential integrity, and all PRD-specific checks. Raw files were preserved unchanged and all corrections were applied to separate cleaned copies.

**Key findings.** Duplicate primary-key records were found in `customers` (**{duplicate_customers:,}**), `billing` (**{duplicate_invoices:,}**) and `payments` (**{duplicate_payments:,}**). Customer-quality issues included **{int(invalid_phone.sum()):,} invalid phone numbers**, **{int(invalid_pin.sum()):,} invalid PIN codes**, **{int(invalid_email.sum()):,} malformed non-null emails**, **{int(age_dob_mismatch.sum()):,} age/DOB inconsistencies** (of which **{int(impossible_age_range.sum()):,}** were outside the 0–120 range), and **{int(mixed_acquisition_date.sum()):,} mixed-format acquisition dates**. Billing contained **{int(future_billing_date.sum()):,} future billing dates**, **{int(negative_invoice.sum()):,} negative totals**, and **{int(incorrect_gst.sum()):,} GST inconsistencies**. Payments contained **{int(broken_payment_invoice_fk.sum()):,} broken non-null invoice references**. Subscriptions contained **{int(null_plan_id.sum()):,} null plan IDs** and **{int(invalid_subscription_status.sum()):,} invalid status values**. `usage_data` contained **{int(precision_anomaly.sum()):,} confirmed corrupted outlier records**, while the IQR rule flagged **{int(iqr_outlier.sum()):,}** statistically unusual usage rows for analytical review rather than automatic deletion.

**Service-quality finding.** **{unresolved_complaints:,} of {len(complaints):,} complaints ({unresolved_share:.2f}%) remain unresolved**, making service recovery an important area for the churn analysis in later phases.

**Referential integrity.** The PRD-required `customer_id` relationships in subscriptions, billing, voice usage, SMS usage and data usage produced **zero broken customer references**. The material referential issue was the known set of broken `payments.invoice_id` links.

**Cleaning actions.** Exact/primary-key duplicates were removed; mixed dates were standardized; canonical city names were restored from the city lookup; ages were recalculated from DOB; unrecoverable malformed contact/PIN values were set to missing; billing GST and totals were recalculated; future billing dates were repaired using the validated 15-day due-date relationship; broken optional invoice references were cleared; invalid subscription statuses were inferred from `end_date`; and confirmed corrupted usage values were set to missing while preserving an outlier flag. Valid business nulls were retained.

**Conclusion.** After cleaning, all targeted post-cleaning validation checks pass. The resulting cleaned datasets are suitable for Phase 2 EDA, while legitimate missing values and statistical usage outliers remain explicitly documented so they are not mistaken for data errors.
"""

display(Markdown(report_md))

with open(
    OUTPUT_DIR / "Phase1_Data_Quality_Report_KashifSiddique.md",
    "w",
    encoding="utf-8",
) as f:
    f.write(report_md)



### NexaTel Phase 1 Data Quality Report

**Scope.** The Phase 1 audit reviewed **24 interconnected tables containing 756,952 raw records**. The assessment covered structure, missing values, duplicates, date/numeric validity, outliers, referential integrity, and all PRD-specific checks. Raw files were preserved unchanged and all corrections were applied to separate cleaned copies.

**Key findings.** Duplicate primary-key records were found in `customers` (**76**), `billing` (**344**) and `payments` (**276**). Customer-quality issues included **228 invalid phone numbers**, **190 invalid PIN codes**, **285 malformed non-null emails**, **38 age/DOB inconsistencies** (of which **23** were outside the 0–120 range), and **380 mixed-format acquisition dates**. Billing contained **229 future billing dates**, **229 negative totals**, and **1,149 GST inconsistencies**. Payments contained **184 broken non-null invoice references**. Subscriptions contained **41 null plan IDs** and **102 invalid status values**. `usage_data` contained **171 confirmed corrupted outlier records**, while the IQR rule flagged **4,480** statistically unusual usage rows for analytical review rather than automatic deletion.

**Service-quality finding.** **7,348 of 16,584 complaints (44.31%) remain unresolved**, making service recovery an important area for the churn analysis in later phases.

**Referential integrity.** The PRD-required `customer_id` relationships in subscriptions, billing, voice usage, SMS usage and data usage produced **zero broken customer references**. The material referential issue was the known set of broken `payments.invoice_id` links.

**Cleaning actions.** Exact/primary-key duplicates were removed; mixed dates were standardized; canonical city names were restored from the city lookup; ages were recalculated from DOB; unrecoverable malformed contact/PIN values were set to missing; billing GST and totals were recalculated; future billing dates were repaired using the validated 15-day due-date relationship; broken optional invoice references were cleared; invalid subscription statuses were inferred from `end_date`; and confirmed corrupted usage values were set to missing while preserving an outlier flag. Valid business nulls were retained.

**Conclusion.** After cleaning, all targeted post-cleaning validation checks pass. The resulting cleaned datasets are suitable for Phase 2 EDA, while legitimate missing values and statistical usage outliers remain explicitly documented so they are not mistaken for data errors.


## 15. Phase 1 requirement checklist

In [22]:
requirements_checklist = pd.DataFrame([
    ["Load all 24 CSV files", "COMPLETE"],
    ["Row count for every table", "COMPLETE"],
    ["Column count for every table", "COMPLETE"],
    ["Data types for every column", "COMPLETE"],
    ["Missing values and % affected", "COMPLETE"],
    ["Duplicate rows / primary keys", "COMPLETE"],
    ["Date validation", "COMPLETE"],
    ["Numeric/amount validation", "COMPLETE"],
    ["data_gb_used outlier check", "COMPLETE"],
    ["Negative total_amount_inr check", "COMPLETE"],
    ["Required referential-integrity checks", "COMPLETE"],
    ["Compare with data_quality_issue_log.csv", "COMPLETE"],
    ["Future billing_date check", "COMPLETE"],
    ["Impossible age check", "COMPLETE"],
    ["Invalid PIN/phone check", "COMPLETE"],
    ["Broken payment invoice_id count", "COMPLETE"],
    ["Unresolved complaint share", "COMPLETE"],
    ["Null subscription plan_id check", "COMPLETE"],
    ["Invalid subscription_status check", "COMPLETE"],
    ["Cleaning code", "COMPLETE"],
    ["Post-cleaning validation", "COMPLETE"],
    ["1-page written Data Quality Report", "COMPLETE"],
], columns=["Phase 1 requirement", "status"])

display(requirements_checklist)


,Phase 1 requirement,status
0,Load all 24 CSV files,COMPLETE
1,Row count for every table,COMPLETE
2,Column count for every table,COMPLETE
3,Data types for every column,COMPLETE
4,Missing values and % affected,COMPLETE
5,Duplicate rows / primary keys,COMPLETE
6,Date validation,COMPLETE
7,Numeric/amount validation,COMPLETE
8,data_gb_used outlier check,COMPLETE
9,Negative total_amount_inr check,COMPLETE


## Phase 1 completion status

**PHASE 1 — COMPLETE ✅**

The notebook now covers every Phase 1 requirement in the NexaTel PRD and includes both the cleaning code and a written Data Quality Report. The cleaned tables exported by this notebook are the approved input for Phase 2 exploratory analysis.
